In [114]:
import numpy as np

# 1D vector operations ---------------------------------------------------------
A = np.random.rand(5, )
B = np.random.rand(5, )

# sum(A)
np.einsum("i -> ", A)

# A*B; elementwise mult
np.einsum("i, i -> i", A, B)

# inner(A, B); dot product (some differences to dot for higher dims...)
np.einsum("i, i -> ", A, B) # same as inner for 1D!

# outer(A, B): outer product
np.einsum("i, j -> ij", A, B)

# 2D array operations ----------------------------------------------------------
C = np.random.rand(5, 5)
D = np.random.rand(5, 5)

# A.T; transpose
np.einsum("ij -> ji", C)

# diag(A); view main diagonal elements
np.einsum("ii -> i", C)

# trace(A); sum main diagonal elements
np.einsum("ii -> ", C)

# sum(A); sum all matrix elements
np.einsum("ij -> ", C)

# sum(A, axis=0); sum along one axis
np.einsum("ij -> j", C) # sum up each column
np.einsum("ij -> i", C) # sum up each row

# A*B; elementwise mult
np.einsum("ij, ij -> ij", C, D)

# A*B.T; elementwise mult with transpose
np.einsum("ij, ji -> ij", C, D) # this kinda just reads in D "in a transpose way"

# dot(A, B); standard matrix mult
np.einsum("ij, jk -> ik", C, D)

# inner(A, B); inner product
np.einsum("ij, kj -> ik", C, D)

# A[:, None] * B; each row of A multiplied by B (elementwise)
np.einsum("ij, kj -> ikj", C, D) # cursed

# A[:, :, None, None] * B; each value of A multiplied by B (elementwise)
np.einsum("ij, kl -> ijkl", C, D) # cursed

# nD arry operations -----------------------------------------------------------

array([[[[6.37934311e-01, 6.89153331e-02, 2.52367049e-01,
          6.18457895e-01, 6.73146791e-01],
         [7.20084222e-01, 2.48675858e-01, 5.68552384e-01,
          4.06515703e-01, 6.83152551e-01],
         [7.63214923e-01, 1.66617289e-01, 8.48967308e-02,
          1.33570027e-01, 3.07361814e-01],
         [7.86485364e-01, 1.04878277e-02, 7.22036132e-02,
          8.11307347e-01, 5.62796957e-01],
         [9.41125761e-02, 8.53741894e-01, 6.18530685e-01,
          1.89537216e-01, 5.88460077e-01]],

        [[6.34823442e-01, 6.85792694e-02, 2.51136388e-01,
          6.15442003e-01, 6.69864209e-01],
         [7.16572751e-01, 2.47463197e-01, 5.65779854e-01,
          4.04533340e-01, 6.79821177e-01],
         [7.59493127e-01, 1.65804784e-01, 8.44827342e-02,
          1.32918676e-01, 3.05862972e-01],
         [7.82650090e-01, 1.04366841e-02, 7.18515143e-02,
          8.07351029e-01, 5.60052493e-01],
         [9.36536387e-02, 8.49578645e-01, 6.15514437e-01,
          1.88612943e-01, 5.855

In [ ]:
# einsum chained contraction performance test. einsum apparently has some problems regarding chained contraction, as it doesn't automatically know which intermediate matrices to "collapse". For example, when chaining multiplications with A, usually and intermediate matrix AA is calculated and then multiplied with the next A. when using the naive chained einsum with ij, jk, kl -> lm for three A's, it will probably broadcast ALL dims, do many multiplications and then sum them at the end

import numpy as np

mat_size = 10
A = np.random.rand(mat_size, mat_size)

print("dot product contraction")
%timeit A.dot(A).dot(A).dot(A)

print("standard einsum, no optimization")
%timeit np.einsum("ij, jk, kl, lm -> im", A, A, A, A)

print("directly using optimization flag")
%timeit np.einsum("ij, jk, kl, lm -> im", A, A, A, A, optimize=True)

print("pre-storing optimization")
opt_path = np.einsum_path("ij, jk, kl, lm -> im", A, A, A, A, optimize=False)[0] # remove info string!
%timeit np.einsum("ij, jk, kl, lm -> im", A, A, A, A, optimize=opt_path)

print("directly specifiying path")
%timeit np.einsum("ij, jk, kl, lm -> im", A, A, A, A, optimize=["einsum_path", (0, 1), (0, 1), (0, 1)])

print("taking operations apart")
def chained_einsum():
    a1 = np.einsum("ij, jk -> ik", A, A)
    a2 = np.einsum("ij, jk -> ik", a1, A)
    a3 = np.einsum("ij, jk -> ik", a2, A)
%timeit chained_einsum()

print("done!")



dot product contraction
2.49 μs ± 18.4 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
standard einsum, no optimization
669 μs ± 2.32 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
directly using optimization flag
133 μs ± 616 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
pre-storing optimization
694 μs ± 4.03 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
directly specifiying path
91.6 μs ± 1.5 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
taking operations apart
10.3 μs ± 53.7 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
done!
